# Tutorial 7 — More data: missing labels, large data sets, free SOCs, doublets and quartets

*Part 4 — Going further · tutorial 7 of 9 · previous: [06 Surface-hopping dynamics with SHARC](06_surface_hopping_with_sharc.ipynb) · next: [08 Oscillator strengths](08_oscillator_strengths.ipynb)*

Real data sets are rarely as tidy as the SO2 set of Tutorial 2. This tutorial covers:

1. **missing labels** and per-frame **weights**;
2. **large data sets** with HDF5 preprocessing;
3. **free SOCs** and data from the original X-MACE;
4. **other multiplicities**: doublets and quartets.

Oscillator strengths, another kind of data, have their own tutorial (8).

**Requirements:** the SO2 data (Tutorial 2). Runtime: about 3 minutes.

In [1]:
import json
import re
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import ase
import ase.io
import matplotlib.pyplot as plt
import numpy as np
import torch

import mace  # noqa: F401

TUTORIAL = Path.cwd()
sys.path.insert(0, str(TUTORIAL / "tools"))
import so2_lvc as lvc

WORK = TUTORIAL / "work"
plt.rcParams.update({"figure.dpi": 100})
np.set_printoptions(precision=4, suppress=True, linewidth=120)
torch.set_default_dtype(torch.float64)  # the trained models are in float64


def strip(line):
    return re.sub(r"^\d{4}-\d\d-\d\d [\d:.]+ [A-Z]+: ", "", line)


def train(name, extra, directory, epochs=10, train_file="../so2_train.xyz"):
    """A short training run in work/<directory> (same settings as Tutorial 3)."""
    run = WORK / directory
    if run.exists():
        shutil.rmtree(run)  # MACE appends to existing log files
    run.mkdir(parents=True)
    command = [
        "mace_run_train", f"--name={name}", f"--train_file={train_file}", "--valid_fraction=0.1",
        "--model=ExcitedMACE", "--states=4 0 3", "--compute_nacs", "--compute_socs", "--compute_dipoles",
        "--E0s=average", "--r_max=5.0", "--hidden_irreps=32x0e + 32x1o", "--MLP_irreps=16x0e",
        "--num_interactions=2", "--correlation=2", "--batch_size=10", f"--max_num_epochs={epochs}",
        "--lr=0.01", "--ema", "--ema_decay=0.99", "--energy_weight=10", "--forces_weight=100",
        "--nacs_weight=10", "--socs_weight=1", "--dipoles_weight=1", "--default_dtype=float64",
        "--device=cpu",
    ] + extra
    result = subprocess.run(command, cwd=run, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(f"training {name} failed")
    return (run / "logs" / f"{name}_run-{_seed(extra)}.log").read_text()


def _seed(extra):
    for item in extra:
        if item.startswith("--seed="):
            return item.split("=")[1]
    return "123"

## 1. Missing labels and weights

Real data sets are rarely complete: some calculations have no NACs (they are
expensive), SOCs were computed only for some geometries, a few points failed. X-MACE
handles this per label: a missing label is filled with zeros and gets weight 0, so
that frame is skipped **for that loss term only**. Per-frame weights work as in
MACE: `config_weight` (all terms) and `config_<property>_weight` in `atoms.info`.

In [2]:
frames = ase.io.read(WORK / "so2_train.xyz", ":")
rng = np.random.default_rng(0)
for atoms in frames:
    if rng.random() < 0.5:
        del atoms.info["REF_socs"]  # no SOCs for half of the frames
    if rng.random() < 0.3:
        del atoms.info["REF_nacs"]  # no NACs for 30 %
    if rng.random() < 0.1:
        atoms.info["config_weight"] = 0.5  # trust these frames less
ase.io.write(WORK / "so2_partial.xyz", frames)
log = train("partial", ["--seed=4"], "partial", epochs=1, train_file="../so2_partial.xyz")
print("\n".join(strip(l) for l in log.splitlines() if "Excited-state labels" in l))

Excited-state labels in train set of head 'Default' (593 configurations): energy=593, forces=593, nacs=421, socs=300, dipoles=593, oscillator_strengths=0
Excited-state labels in valid set of head 'Default' (65 configurations): energy=65, forces=65, nacs=44, socs=28, dipoles=65, oscillator_strengths=0


## 2. Large data sets: HDF5

For tens of thousands of frames, reading XYZ files and building graphs at every
start becomes slow. `mace_prepare_data` (the script `mace/cli/preprocess_data.py`) converts the data (including all
excited-state labels) to HDF5 once, and computes the statistics (E0s, scale, one
shift per state):

In [3]:
h5 = WORK / "h5"
subprocess.run(
    ["mace_prepare_data", f"--train_file={WORK / 'so2_train.xyz'}", "--valid_fraction=0.1",
     f"--h5_prefix={h5}/", "--r_max=5.0", "--E0s=average", "--compute_statistics",
     "--states=4 0 3", "--compute_nacs", "--compute_socs", "--compute_dipoles", "--num_process=1", "--seed=1"],
    check=True, capture_output=True,
)
print(json.loads((h5 / "statistics.json").read_text()))
print(sorted(p.relative_to(h5).as_posix() for p in h5.rglob("*.h5")))

{'atomic_energies': '{8: 1.7233899255717404, 16: 0.8616949627858713}', 'avg_num_neighbors': 2.0, 'mean': [-1.2002489566802979, -0.05021153762936592, 0.1691104769706726, 1.3389277458190918, -0.252568781375885, -0.06598671525716782, 0.06097735837101936], 'std': 3.748506546020508, 'atomic_numbers': '[8, 16]', 'r_max': 5.0}
['train/train_0.h5', 'val/val_0.h5']


Train from the HDF5 files with the same model options:

```bash
mace_run_train --model=ExcitedMACE --states="4 0 3" --compute_nacs --compute_socs \
  --compute_dipoles --train_file=h5/train/train_0.h5 --valid_file=h5/val/val_0.h5 \
  --statistics_file=h5/statistics.json ...
```

## 3. Free SOCs and data from the original X-MACE

`--soc_mode=free` learns the SOCs as `--soc_num` invariant numbers per frame, without
any assumption about their structure. Use it for SOC data that is not in SHARC's
convention and cannot be converted, or for X-MACE data sets. The original X-MACE
stored the upper triangle of the M_S-resolved SOC matrix as real numbers; with
`--soc_num` equal to its length, the SHARC interface still builds the full
Hermitian matrix. The price: no symmetry constraints (no selection rules, no
rotation covariance), so more data is needed.

X-MACE command lines keep working: `--n_energies=3 --compute_nacs --compute_socs
--soc_num=...` without `--states` trains states of one multiplicity with free SOCs.

In [4]:
from types import SimpleNamespace

from mace.data.excited import excited_spec_from_args

common = dict(model="ExcitedMACE", compute_nacs=True, compute_socs=True, compute_dipoles=True, nac_num=None,
              dipole_num=None, nacs_format="raw", nacs_key="REF_nacs")
for options in (
    dict(states="4 0 3", n_energies=None, soc_num=None, soc_mode="auto"),
    dict(states="4 0 3", n_energies=None, soc_num=78, soc_mode="free"),
    dict(states=None, n_energies=3, soc_num=3, soc_mode="auto"),
):
    spec = excited_spec_from_args(SimpleNamespace(**common, **options))
    print(f"{options} ->\n   {spec.describe()}")

{'states': '4 0 3', 'n_energies': None, 'soc_num': None, 'soc_mode': 'auto'} ->
   7 states (4 singlets, 3 triplets), NACs=9 (raw), SOCs=15 coupling vectors (spin_orbit), dipoles=16
{'states': '4 0 3', 'n_energies': None, 'soc_num': 78, 'soc_mode': 'free'} ->
   7 states (4 singlets, 3 triplets), NACs=9 (raw), SOCs=78, dipoles=16
{'states': None, 'n_energies': 3, 'soc_num': 3, 'soc_mode': 'auto'} ->
   3 states, NACs=3 (raw), SOCs=3, dipoles=6


## 4. Doublets and quartets

Radicals have doublet and quartet states: `--states="0 3 0 2"` means 3 doublets and 2
quartets. Everything works as for singlets and triplets: each doublet has two M_S
components (±1/2), each quartet four; NACs couple D–D and Q–Q; SOC vectors couple
D–D, D–Q and Q–Q pairs. A quick look with an untrained model. Its energy readouts
start at zero, so every state starts on the backbone's ground-state surface; random
readout weights make the states differ, as they would after training.

In [5]:
from e3nn import o3

from mace import modules, tools
from mace.calculators.sharc_calculator import SharcCalculator
from mace.modules.spin_states import describe_states, n_mstates, spin_orbit_pairs

radical_states = [0, 3, 0, 2]
z_table = tools.AtomicNumberTable([1, 6, 8])
torch.manual_seed(0)
radical = modules.ExcitedMACE(
    r_max=4.0, num_bessel=8, num_polynomial_cutoff=5, max_ell=2,
    interaction_cls=modules.interaction_classes["RealAgnosticResidualInteractionBlock"],
    interaction_cls_first=modules.interaction_classes["RealAgnosticInteractionBlock"],
    num_interactions=2, num_elements=3, hidden_irreps=o3.Irreps("16x0e + 16x1o"),
    MLP_irreps=o3.Irreps("16x0e"), atomic_energies=np.zeros(3), avg_num_neighbors=3.0,
    atomic_numbers=z_table.zs, correlation=2, gate=torch.nn.functional.silu,
    n_energies=sum(radical_states), states=radical_states,
    compute_nacs=True, compute_socs=True, soc_mode="spin_orbit",
)
with torch.no_grad():  # stand-in for training: random energy readouts
    for readout in radical.readouts:
        last = getattr(readout, "linear_2", None) or readout.linear
        last.weight.normal_(0.0, 0.3)
torch.save(radical, WORK / "radical.model")
print(describe_states(radical_states), "|", n_mstates(radical_states), "M_S states |",
      radical.nac_num, "NACs |", radical.soc_vector_num, "SOC vectors:", spin_orbit_pairs(radical_states))

hco = ase.Atoms("HCO", positions=[[0.0, 1.08, -0.55], [0.0, 0.0, 0.0], [0.0, 0.1, 1.18]])  # formyl radical
sharc = SharcCalculator(hco.get_chemical_symbols(), str(WORK / "radical.model"), states=radical_states,
                        properties=["energy", "forces", "nacs", "socs"])
qm = sharc.calculate(hco.positions / lvc.BOHR_TO_ANGSTROM)
print("h:", qm["h"].shape, "| Hermitian:", np.allclose(qm["h"], qm["h"].conj().T))
print("doublet energies (Hartree), M_S = -1/2 then +1/2:", np.diag(qm["h"]).real[:6].round(4))

3 doublets, 2 quartets | 14 M_S states | 4 NACs | 10 SOC vectors: [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]


h: (14, 14) | Hermitian: True
doublet energies (Hartree), M_S = -1/2 then +1/2: [0.0014 0.0023 0.0011 0.0014 0.0023 0.0011]


In SHARC set `nstates 0 3 0 2` and one charge per multiplicity, e.g. `charge 0 0 0 0`.

**Next:** [Tutorial 8](08_oscillator_strengths.ipynb) learns oscillator strengths, also
from data that has nothing else.